# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
from google.colab import drive

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

In [ ]:
# Cấu hình
warnings.filterwarnings('ignore') # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option('display.float_format', '{:.2f}'.format) # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update({
    'figure.dpi'         : 150,
    'axes.grid'          : True,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'font.family'        : 'DejaVu Sans',
}) # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    'Good'                            : '#00E400',
    'Moderate'                        : '#FFFF00',
    'Unhealthy for Sensitive Groups'  : '#FF7E00',
    'Unhealthy'                       : '#FF0000',
    'Very Unhealthy'                  : '#8F3F97',
    'Hazardous'                       : '#7E0023',
}

AQI_BINS   = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}

# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:    return 'Good'
    elif value <= 100: return 'Moderate'
    elif value <= 150: return 'Unhealthy for Sensitive Groups'
    elif value <= 200: return 'Unhealthy'
    elif value <= 300: return 'Very Unhealthy'
    else:              return 'Hazardous'

# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [val / MAX_AQI for val in AQI_BINS] # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]] # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(zip(positions, gradient_colors)) # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list('AQI_gradient', color_mapping) # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# Cái này để cho nhóm trưởng lưu Files nha (Mặc kệ nó đi)
# drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

In [ ]:
folder_ID = "1b8LGMtOwMrj6FDMODA8-rJNq0cKXlgBA"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output='data', quiet=False)

X_train = pd.read_csv('X_train.csv', index_col='date', parse_dates=True)
X_test  = pd.read_csv('X_test.csv',  index_col='date', parse_dates=True)
y_train = pd.read_csv('y_train.csv', index_col='date', parse_dates=True)
y_test  = pd.read_csv('y_test.csv',  index_col='date', parse_dates=True)

y_train_reg = y_train['target_reg_tomorrow']
y_test_reg  = y_test['target_reg_tomorrow']

In [ ]:
# Hàm đánh giá
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    print(f'{name:<25} MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'MAE':mae, 'RMSE':rmse, 'R²':r2, 'MAPE':mape}

## **03. Model Regression**

### **3.1. Ridge Regression (Baseline)**

### **3.2. Decision Tree**

### **3.3. Random Forest**

### **3.4. XGBoost - LightGBM**

#### **Kết hợp XGBoost và LightGBM**

### **3.5. SARIMA**

### **3.6. So sánh tổng hợp**

### **3.7. Đánh giá Best Model**

### **3.8. Phân tích SARIMA với Machine Learning**

### **3.9. Tổng kết**